In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000000.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000002.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000004.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000006.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000026.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000028.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000029.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000030.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000031.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000035.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000046.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISIC_0000054.jpg
/kaggle/input/datasets/iamtrina/isic-images-full/images_full/ISI

In [2]:
import os, sys, subprocess, warnings, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
try:
    from captum.attr import IntegratedGradients
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'captum', '-q'])
    from captum.attr import IntegratedGradients

# ---------------- CONFIG ----------------
IMAGE_FOLDER = '/kaggle/input/datasets/iamtrina/isic-images-full/images_full'
TRAIN_META   = '/kaggle/input/datasets/sarifasiddikaa/isic-dicm-17k/train-set-metadata.csv'
PRED_CSV     = '/kaggle/input/datasets/sarifasiddikaa/final-results-15k/final_predictions_per_sample.csv'
CKPT_DIR     = '/kaggle/input/datasets/sarifasiddikaa/final-results-15k'
OUT_DIR      = '/kaggle/working'
N_PER_CLASS  = 50
IG_STEPS     = 20
SEEDS        = [1, 2, 3, 4, 5]
T0 = time.time()
def log(msg): print(f'[{(time.time()-T0)/60:6.1f} min] {msg}', flush=True)
# ----------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
cat_cols = ['sex', 'anatom_site_general', 'personal_hx_mm', 'family_hx_mm']
numeric_cols = ['age_approx', 'clin_size_long_diam_mm']
def S(x): return x.map(lambda v: 'nan' if pd.isna(v) else str(v))

# ================= PART 1: SUBGROUP + DISAGREEMENT AUDIT (no retraining) =================
_pred = pd.read_csv(PRED_CSV)
_meta = pd.read_csv(TRAIN_META).drop_duplicates('isic_id')
_a = _pred.merge(_meta, on='isic_id', how='left')
assert _a['class'].notna().all(), 'some predictions have no metadata row'
_a['y'] = _a['true_label'].astype(int)
_a['pF'] = _a['fusion_prob_1']; _a['pM'] = _a['mb_prob_1']
_fields = ['sex', 'age_approx', 'anatom_site_general', 'clin_size_long_diam_mm', 'personal_hx_mm', 'family_hx_mm']
_a['n_fields'] = _a[_fields].notna().sum(axis=1)
_a['completeness'] = pd.cut(_a['n_fields'], [-1, 2, 3, 4, 6], labels=['0-2 fields', '3 fields', '4 fields', '5-6 fields']).astype(str)
_a['patient_id_status'] = np.where(_a['patient_id'].notna(), 'patient ID known', 'patient ID missing')
_a['age_group'] = pd.cut(_a['age_approx'], [0, 40, 60, 120], labels=['<=40', '41-60', '>60']).astype(str).replace('nan', 'age missing')
_a['sex_group'] = _a['sex'].fillna('sex missing')
_top_sites = _a['anatom_site_general'].value_counts().head(5).index.tolist()
_a['site_group'] = _a['anatom_site_general'].map(lambda v: 'site missing' if pd.isna(v) else (v if v in _top_sites else 'other sites'))
_a['class_group'] = np.where(_a['y'] == 1, 'melanoma', 'non-melanoma')
_a['predF'] = (_a['pF'] >= 0.5).astype(int); _a['predM'] = (_a['pM'] >= 0.5).astype(int)
_a['okF'] = _a['predF'] == _a['y']; _a['okM'] = _a['predM'] == _a['y']
_a['disagree'] = _a['predF'] != _a['predM']
_a['confwrongF'] = (~_a['okF']) & ((_a['pF'] >= 0.9) | (_a['pF'] <= 0.1))
_a['confwrongM'] = (~_a['okM']) & ((_a['pM'] >= 0.9) | (_a['pM'] <= 0.1))

def _auc(y, p): return roc_auc_score(y, p) if len(set(y)) == 2 else np.nan
def _sens_spec(y, pred):
    s = (pred[y == 1] == 1).mean() if (y == 1).any() else np.nan
    c = (pred[y == 0] == 0).mean() if (y == 0).any() else np.nan
    return s, c
sub_rows = []
for gcol in ['completeness', 'patient_id_status', 'age_group', 'sex_group', 'site_group', 'class_group']:
    for gval, g in _a.groupby(gcol):
        per = []
        for sd, gs in g.groupby('seed'):
            y = gs['y'].values
            r = {'n': len(gs), 'mel_frac': y.mean()}
            for tag, p, pr in [('Fusion', gs['pF'].values, gs['predF'].values), ('MetaBlock', gs['pM'].values, gs['predM'].values)]:
                se, sp = _sens_spec(y, pr)
                r.update({f'{tag}_AUROC': _auc(y, p), f'{tag}_acc': (pr == y).mean(), f'{tag}_sens': se, f'{tag}_spec': sp})
            per.append(r)
        P = pd.DataFrame(per); row = {'grouping': gcol, 'group': gval, 'mean_n_per_run': round(P['n'].mean()), 'melanoma_%': round(100 * P['mel_frac'].mean(), 1)}
        for k in [c for c in P.columns if c not in ('n', 'mel_frac')]:
            row[k] = f'{P[k].mean():.3f} ± {P[k].std(ddof=1):.3f}' if P[k].notna().sum() > 1 else (f'{P[k].mean():.3f}' if P[k].notna().any() else 'n/a')
        row['note'] = 'small group: interpret cautiously' if P['n'].mean() < 100 else ''
        sub_rows.append(row)
sub = pd.DataFrame(sub_rows); sub.to_csv(f'{OUT_DIR}/isic_subgroup_metrics.csv', index=False)
print('\n===== SUBGROUP METRICS (mean ± SD over 5 runs) =====')
print(sub[['grouping', 'group', 'mean_n_per_run', 'melanoma_%', 'Fusion_AUROC', 'MetaBlock_AUROC', 'MetaBlock_sens', 'MetaBlock_spec', 'note']].to_string(index=False))

aud = []
def _audit(label, g):
    per = []
    for sd, gs in g.groupby('seed'):
        d = gs[gs['disagree']]
        per.append({'n': len(gs), 'disagree_%': 100 * gs['disagree'].mean(), 'n_disagree': len(d),
                    'MB_correct_when_disagree_%': 100 * d['okM'].mean() if len(d) else np.nan,
                    'both_wrong_%': 100 * ((~gs['okF']) & (~gs['okM'])).mean(),
                    'conf_wrong_Fusion_%': 100 * gs['confwrongF'].mean(), 'conf_wrong_MetaBlock_%': 100 * gs['confwrongM'].mean()})
    P = pd.DataFrame(per); r = {'subset': label, 'mean_n_per_run': round(P['n'].mean())}
    for k in [c for c in P.columns if c != 'n']: r[k] = f'{P[k].mean():.2f} ± {P[k].std(ddof=1):.2f}'
    aud.append(r)
_audit('ALL test images', _a)
for gcol in ['completeness', 'patient_id_status', 'age_group', 'site_group', 'class_group']:
    for gval, g in _a.groupby(gcol): _audit(f'{gcol}: {gval}', g)
aud = pd.DataFrame(aud); aud.to_csv(f'{OUT_DIR}/isic_disagreement_audit.csv', index=False)
print('\n===== FUSION vs METABLOCK DISAGREEMENT AUDIT =====')
print(aud.to_string(index=False))
print('\nPART 1 DONE.\n')

# ================= PART 2: INTEGRATED GRADIENTS SENSITIVITY (5 seeds, ~45-60 min) =================
df = pd.read_csv(TRAIN_META)
mel_full = df[df['class'] == 1]
non_mel_full = df[df['class'] == 0].sample(n=len(mel_full), random_state=42)
full_subset = pd.concat([mel_full, non_mel_full]).reset_index(drop=True)
full_subset['patient_id'] = S(full_subset['patient_id'])
full_subset.loc[full_subset['patient_id'] == 'nan', 'patient_id'] = 'unknown_' + full_subset.index[full_subset['patient_id'] == 'nan'].astype(str)
def check_image(isic_id):
    p = f"{IMAGE_FOLDER}/{isic_id}.jpg"
    if not os.path.exists(p): return isic_id
    try: Image.open(p).convert('RGB'); return None
    except Exception: return isic_id
log('checking images (parallel) ...')
with ThreadPoolExecutor(16) as ex: res_chk = list(ex.map(check_image, full_subset['isic_id'].tolist()))
corrupted = [r for r in res_chk if r is not None]
full_subset = full_subset[~full_subset['isic_id'].isin(corrupted)].reset_index(drop=True)
log(f'Usable total: {len(full_subset)} | Corrupted/missing: {len(corrupted)}')

def encode_metadata(df_in, medians, reference_columns=None):
    df_cat = pd.get_dummies(df_in[cat_cols].apply(S))
    df_num = df_in[numeric_cols].fillna(medians)
    out = pd.concat([df_num, df_cat], axis=1)
    if reference_columns is not None: out = out.reindex(columns=reference_columns, fill_value=0)
    return out

IMAGE_LOOKUP = {f: os.path.join(IMAGE_FOLDER, f) for f in os.listdir(IMAGE_FOLDER)}
class DS(Dataset):
    def __init__(self, df, meta, labels):
        self.df = df.reset_index(drop=True); self.meta = meta.reset_index(drop=True); self.labels = labels
        self.cache = {}; self.resize = transforms.Resize((224, 224))
        self.tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        n = self.df.loc[i, 'img_id']
        if n not in self.cache: self.cache[n] = self.resize(Image.open(IMAGE_LOOKUP[n]).convert('RGB'))
        return self.tf(self.cache[n].copy()), torch.tensor(self.meta.iloc[i].values.astype('float32')), torch.tensor(int(self.labels[i]), dtype=torch.long)

def backbone():
    eff = models.efficientnet_b3(weights=None); return nn.Sequential(eff.features, eff.avgpool)
class FusionModel(nn.Module):
    def __init__(self, meta_dim, num_classes=2):
        super().__init__()
        self.image_branch = backbone()
        self.meta_branch = nn.Sequential(nn.Linear(meta_dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.classifier = nn.Sequential(nn.Linear(1536 + 32, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, num_classes))
    def forward(self, img, meta):
        return self.classifier(torch.cat([self.image_branch(img).flatten(1), self.meta_branch(meta)], dim=1))
class MetaBlockModel(nn.Module):
    def __init__(self, meta_dim, num_classes=2):
        super().__init__()
        self.image_branch = backbone()
        self.meta_gate = nn.Sequential(nn.Linear(meta_dim, 1536)); self.meta_bias = nn.Sequential(nn.Linear(meta_dim, 1536))
        self.classifier = nn.Sequential(nn.Linear(1536, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, num_classes))
    def forward(self, img, meta):
        i = self.image_branch(img).flatten(1); g = torch.sigmoid(self.meta_gate(meta)); b = torch.tanh(self.meta_bias(meta))
        return self.classifier(i * g + b), g
def load(cls, path, dim, name):
    m = cls(dim).to(device); r = m.load_state_dict(torch.load(path, map_location=device), strict=False)
    if r.missing_keys or r.unexpected_keys: print(f'  [WARNING] {name}: missing {r.missing_keys} unexpected {r.unexpected_keys}')
    else: print(f'  {name} checkpoint loaded cleanly')
    return m.eval()

pred_ref = pd.read_csv(PRED_CSV)
rows, per_sample = [], []
VARIANTS = [('zero', 'softmax'), ('zero', 'logit'), ('mean', 'softmax'), ('mean', 'logit')]
IMG_DIM = 3 * 224 * 224
def spear(a, b): return spearmanr(a, b)[0]

for seed in SEEDS:
    log(f'=== Seed {seed} ===')
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tv_idx, te_idx = next(s1.split(full_subset, full_subset['class'], groups=full_subset['patient_id']))
    tv = full_subset.iloc[tv_idx].reset_index(drop=True); test_df = full_subset.iloc[te_idx].reset_index(drop=True)
    s2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, va_idx = next(s2.split(tv, tv['class'], groups=tv['patient_id']))
    train_df = tv.iloc[tr_idx].reset_index(drop=True)
    same = set(test_df['isic_id']) == set(pred_ref[pred_ref['seed'] == seed]['isic_id'])
    print('Test set identical to the saved predictions:', same, f'({len(test_df)} images)')
    assert same, 'split could not be reproduced'
    for d in [train_df, test_df]:
        d['img_id'] = d['isic_id'] + '.jpg'
        for c in cat_cols: d[c] = S(d[c])
    med = train_df[numeric_cols].median()
    tm = encode_metadata(train_df, med); rc = tm.columns; sm = encode_metadata(test_df, med, rc); META_DIM = len(rc)
    labels = test_df['class'].values.astype(int)
    ds = DS(test_df, sm, labels); loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    fusion = load(FusionModel, f'{CKPT_DIR}/fusion_seed{seed}.pth', META_DIM, 'Fusion')
    mb = load(MetaBlockModel, f'{CKPT_DIR}/metablock_seed{seed}.pth', META_DIM, 'MetaBlock')

    log('running all test images through the models ...')
    P = {'full': [], 'img': [], 'meta': []}; G = []
    with torch.no_grad():
        for img, meta, y in loader:
            img, meta, y = img.to(device), meta.to(device), y.to(device)
            for k, (i_, m_) in {'full': (img, meta), 'img': (img, torch.zeros_like(meta)), 'meta': (torch.zeros_like(img), meta)}.items():
                P[k].append(torch.softmax(fusion(i_, m_), 1).gather(1, y[:, None])[:, 0].cpu())
            G.append(mb(img, meta)[1].mean(1).cpu())
    pIM, pI, pM = [torch.cat(P[k]).numpy() for k in ['full', 'img', 'meta']]; gate = torch.cat(G).numpy()
    rI = pIM - pM; shap_img = 0.5 * ((pI - 0.5) + (pIM - pM))
    rho_shap_all, rho_gate_all = spear(shap_img, rI), spear(gate, rI)
    print(f'  all test images: mean r_I {rI.mean():.4f}, rho(Shapley, r_I) {rho_shap_all:.4f}, rho(gate, r_I) {rho_gate_all:.4f}, accuracy {(pIM >= 0.5).mean():.4f}')

    log('starting Integrated Gradients (8 runs) ...')
    rs = np.random.RandomState(seed); i1, i0 = np.where(labels == 1)[0], np.where(labels == 0)[0]
    sel = np.sort(np.concatenate([rs.choice(i1, min(N_PER_CLASS, len(i1)), replace=False), rs.choice(i0, min(N_PER_CLASS, len(i0)), replace=False)]))
    sel_lab = labels[sel]
    sub_loader = DataLoader(torch.utils.data.Subset(ds, sel.tolist()), batch_size=4, shuffle=False, num_workers=0)
    img_mean = torch.stack([ds[i][0] for i in sel]).mean(0, keepdim=True)
    meta_mean = torch.tensor(tm.values.astype('float32')).mean(0, keepdim=True)

    def ig_variant(model, base, target):
        def fwd(img, meta):
            out = model(img, meta)
            return torch.softmax(out, 1) if target == 'softmax' else out
        ig = IntegratedGradients(fwd); si, sm_ = [], []
        for img, meta, y in sub_loader:
            img, meta, y = img.to(device), meta.to(device), y.to(device)
            if base == 'zero': b_img, b_meta = torch.zeros_like(img), torch.zeros_like(meta)
            else: b_img, b_meta = img_mean.to(device).expand_as(img).contiguous(), meta_mean.to(device).expand_as(meta).contiguous()
            a_i, a_m = ig.attribute((img, meta), baselines=(b_img, b_meta), target=y, n_steps=IG_STEPS, internal_batch_size=4)
            si.append(a_i.abs().flatten(1).sum(1).double().cpu()); sm_.append(a_m.abs().flatten(1).sum(1).double().cpu())
            del a_i, a_m
            if device.type == 'cuda': torch.cuda.empty_cache()
        si, sm_ = torch.cat(si).numpy(), torch.cat(sm_).numpy()
        raw = si / (si + sm_ + 1e-300)
        a, b = si / IMG_DIM, sm_ / META_DIM
        perdim = a / (a + b + 1e-300)
        return raw, perdim

    torch.manual_seed(seed); rand_model = FusionModel(META_DIM).to(device).eval()
    for base, target in VARIANTS:
        raw, pdim = ig_variant(fusion, base, target); raw_r, pdim_r = ig_variant(rand_model, base, target)
        r = {'seed': seed, 'baseline': base, 'target': target, 'n_mel': int((sel_lab == 1).sum()), 'n_non': int((sel_lab == 0).sum()),
             'mean_raw_fraction': raw.mean(), 'mean_perdim_fraction': pdim.mean(),
             'perdim_fraction_mel': pdim[sel_lab == 1].mean(), 'perdim_fraction_non': pdim[sel_lab == 0].mean(),
             'rho_raw_vs_rI': spear(raw, rI[sel]), 'rho_perdim_vs_rI': spear(pdim, rI[sel]),
             'mean_raw_fraction_RANDOM': raw_r.mean(), 'mean_perdim_fraction_RANDOM': pdim_r.mean(),
             'rho_trained_vs_random_raw': spear(raw, raw_r), 'rho_trained_vs_random_perdim': spear(pdim, pdim_r),
             'rho_shapley_all': rho_shap_all, 'rho_gate_all': rho_gate_all, 'mean_rI_all': rI.mean(),
             'rho_shapley_subset': spear(shap_img[sel], rI[sel]), 'rho_gate_subset': spear(gate[sel], rI[sel])}
        rows.append(r); log('variant done'); print(f"  IG {base:4s}/{target:7s}: raw {r['mean_raw_fraction']:.4f} | per-dim {r['mean_perdim_fraction']:.3f} | rho(raw,r_I) {r['rho_raw_vs_rI']:+.3f} | rho(per-dim,r_I) {r['rho_perdim_vs_rI']:+.3f} | trained-vs-random {r['rho_trained_vs_random_raw']:+.3f}")
    for j in range(len(test_df)):
        per_sample.append({'seed': seed, 'isic_id': test_df.loc[j, 'isic_id'], 'true_label': labels[j], 'p_full': pIM[j], 'p_img_only': pI[j],
                           'p_meta_only': pM[j], 'rI': rI[j], 'shap_img': shap_img[j], 'gate_mean': gate[j]})
    pd.DataFrame(rows).to_csv(f'{OUT_DIR}/isic_ig_sensitivity_per_seed.csv', index=False)
    pd.DataFrame(per_sample).to_csv(f'{OUT_DIR}/isic_attribution_per_sample_perrun_encoding.csv', index=False)
    del fusion, mb, rand_model; torch.cuda.empty_cache()

res = pd.DataFrame(rows)
print('\n===== IG SENSITIVITY: mean ± SD over runs =====')
cols = ['mean_raw_fraction', 'mean_perdim_fraction', 'perdim_fraction_mel', 'perdim_fraction_non', 'rho_raw_vs_rI', 'rho_perdim_vs_rI',
        'mean_raw_fraction_RANDOM', 'rho_trained_vs_random_raw', 'rho_trained_vs_random_perdim']
out = res.groupby(['baseline', 'target'])[cols].agg(lambda v: f'{v.mean():.3f} ± {v.std(ddof=1):.3f}')
print(out.T.to_string())
chk = res.drop_duplicates('seed')
print(f"\nrho(Shapley, r_I) all images: {chk['rho_shapley_all'].mean():.3f} ± {chk['rho_shapley_all'].std(ddof=1):.3f}")
print(f"rho(gate, r_I) all images:    {chk['rho_gate_all'].mean():.3f} ± {chk['rho_gate_all'].std(ddof=1):.3f}")
print('\nDONE. Files in', OUT_DIR)

Device: cuda

===== SUBGROUP METRICS (mean ± SD over 5 runs) =====
         grouping              group  mean_n_per_run  melanoma_%  Fusion_AUROC MetaBlock_AUROC MetaBlock_sens MetaBlock_spec                              note
     completeness         0-2 fields             190        55.5 0.904 ± 0.020   0.915 ± 0.019  0.888 ± 0.043  0.849 ± 0.035                                  
     completeness           3 fields            1747        56.1 0.833 ± 0.011   0.834 ± 0.007  0.879 ± 0.028  0.614 ± 0.034                                  
     completeness           4 fields             218        30.0 0.771 ± 0.025   0.793 ± 0.021  0.671 ± 0.076  0.749 ± 0.037                                  
     completeness         5-6 fields             864        42.2 0.795 ± 0.020   0.812 ± 0.020  0.699 ± 0.041  0.750 ± 0.035                                  
patient_id_status   patient ID known            1517        38.3 0.829 ± 0.014   0.835 ± 0.014  0.720 ± 0.041  0.785 ± 0.033              